
# Challenge #6: Spatial Route

>We hope you enjoyed last week's challenge.   For the sixth challenge lets look at some of the geospatial capabilities in Alteryx.

>The challenge for week 6 will focus on analyzing distance traveled by sales reps.

>Sales reps are travelling all over the US. The data contained in the workflow details the travel paths for 7 Reps to 7 different cities. The travel route is detailed as well. The objective of this challenge is to **determine which Rep has logged the most miles. Please include the route traveled as a spatial object in the output.**

>We have listed this as an intermediate challenge since not everyone is familiar with the Spatial tools.  As always, we love to hear your comments. We hope you are having fun with the challenges!

In [155]:
# import and preview output
 
import pandas as pd
from math import sqrt, cos, radians, pi, asin
# import geopy

output = pd.read_csv('output.csv', index_col = False)
df_output = pd.DataFrame(output)
df_output.head(25)

,REP,LengthMi
0,Diana,9077.844095
1,Cathy,6777.003786
2,Dan,6220.995173
3,James,6157.018626
4,Steve,4250.389398
5,Scott,4178.885363
6,Mark,3210.917267


In [156]:
# import and preview input

input = pd.read_csv('input.csv', index_col = False)
df = pd.DataFrame(input)
df.head(10)

,Airport City,Centroid,REP,TripOrder
0,"Hartford-West Hartford-East Hartford, CT Metro","{ ""type"": ""Point"", ""coordinates"": [ -72.577485...",Cathy,1
1,"Indianapolis-Carmel-Anderson, IN Metro","{ ""type"": ""Point"", ""coordinates"": [ -86.206137...",Cathy,2
2,"Springfield, MA Metro","{ ""type"": ""Point"", ""coordinates"": [ -72.646741...",Cathy,3
3,"Reno, NV Metro","{ ""type"": ""Point"", ""coordinates"": [ -119.66065...",Cathy,4
4,"Cleveland-Elyria, OH Metro","{ ""type"": ""Point"", ""coordinates"": [ -81.683882...",Cathy,5
5,"Knoxville, TN Metro","{ ""type"": ""Point"", ""coordinates"": [ -84.138485...",Cathy,6
6,"Madison, WI Metro","{ ""type"": ""Point"", ""coordinates"": [ -89.591695...",Cathy,7
7,"Phoenix-Mesa-Scottsdale, AZ Metro","{ ""type"": ""Point"", ""coordinates"": [ -112.07055...",Dan,1
8,"Miami-Fort Lauderdale-West Palm Beach, FL Metro","{ ""type"": ""Point"", ""coordinates"": [ -80.497102...",Dan,2
9,"Lexington-Fayette, KY Metro","{ ""type"": ""Point"", ""coordinates"": [ -84.432749...",Dan,3


In [157]:
df["lat"] = df["Centroid"].str.extract(r"([-\d\.]+),").astype(float)
df["lon"] = df["Centroid"].str.extract(r", ([-\d\.]+)").astype(float)
df.head(5)

,Airport City,Centroid,REP,TripOrder,lat,lon
0,"Hartford-West Hartford-East Hartford, CT Metro","{ ""type"": ""Point"", ""coordinates"": [ -72.577485...",Cathy,1,-72.577485,41.735909
1,"Indianapolis-Carmel-Anderson, IN Metro","{ ""type"": ""Point"", ""coordinates"": [ -86.206137...",Cathy,2,-86.206137,39.747431
2,"Springfield, MA Metro","{ ""type"": ""Point"", ""coordinates"": [ -72.646741...",Cathy,3,-72.646741,42.230068
3,"Reno, NV Metro","{ ""type"": ""Point"", ""coordinates"": [ -119.66065...",Cathy,4,-119.660656,40.638609
4,"Cleveland-Elyria, OH Metro","{ ""type"": ""Point"", ""coordinates"": [ -81.683882...",Cathy,5,-81.683882,41.375258


In [158]:
def get_distance(lat1, lon1, lat2, lon2): 
    R = 6371  # radius of the earth in km
    x = (radians(lon2) - radians(lon1)) * cos(0.5 * (radians(lat2) + radians(lat1)))
    y = radians(lat2) - radians(lat1)
    d = R * sqrt(x*x + y*y)
    return d

In [159]:
print(get_distance(1, 2, 3, 4))

314.41096674274627


In [160]:
def get_distance2(lat1, lon1, lat2, lon2): 
    R = 6371  # radius of the earth in km
    P = pi / 180
    a = 0.5 - cos((lat2 - lat1) * P) / 2 + cos(lat1 * P) * cos(lat2 * P) * (1 - cos((lon2 - lon1) * P)) / 2
    return 2 * R * asin(sqrt(a))

In [161]:
print(get_distance2(1, 2, 3, 4))

314.4029510236166


In [162]:
# distance =
# if TripOrder != 1, get_distance(this_row.lat, this_row.lon, last_row.lat, last_row.lon)

# group by REP, sum distance
# sort by distance descending

In [163]:
df["prev_lat"] = df["lat"].shift(1)
df["prev_lon"] = df["lon"].shift(1)
df.head()

,Airport City,Centroid,REP,TripOrder,lat,lon,prev_lat,prev_lon
0,"Hartford-West Hartford-East Hartford, CT Metro","{ ""type"": ""Point"", ""coordinates"": [ -72.577485...",Cathy,1,-72.577485,41.735909,NaN,NaN
1,"Indianapolis-Carmel-Anderson, IN Metro","{ ""type"": ""Point"", ""coordinates"": [ -86.206137...",Cathy,2,-86.206137,39.747431,-72.577485,41.735909
2,"Springfield, MA Metro","{ ""type"": ""Point"", ""coordinates"": [ -72.646741...",Cathy,3,-72.646741,42.230068,-86.206137,39.747431
3,"Reno, NV Metro","{ ""type"": ""Point"", ""coordinates"": [ -119.66065...",Cathy,4,-119.660656,40.638609,-72.646741,42.230068
4,"Cleveland-Elyria, OH Metro","{ ""type"": ""Point"", ""coordinates"": [ -81.683882...",Cathy,5,-81.683882,41.375258,-119.660656,40.638609


In [164]:
df['distance1'] = df.apply(lambda row: get_distance(row.lat, row.lon, row.prev_lat, row.prev_lon), axis=1)
df['distance2'] = df.apply(lambda row: get_distance2(row.lat, row.lon, row.prev_lat, row.prev_lon), axis=1)
df.head(10)

,Airport City,Centroid,REP,TripOrder,lat,lon,prev_lat,prev_lon,distance1,distance2
0,"Hartford-West Hartford-East Hartford, CT Metro","{ ""type"": ""Point"", ""coordinates"": [ -72.577485...",Cathy,1,-72.577485,41.735909,NaN,NaN,NaN,NaN
1,"Indianapolis-Carmel-Anderson, IN Metro","{ ""type"": ""Point"", ""coordinates"": [ -86.206137...",Cathy,2,-86.206137,39.747431,-72.577485,41.735909,1515.983515,1515.759492
2,"Springfield, MA Metro","{ ""type"": ""Point"", ""coordinates"": [ -72.646741...",Cathy,3,-72.646741,42.230068,-86.206137,39.747431,1508.586751,1508.239316
3,"Reno, NV Metro","{ ""type"": ""Point"", ""coordinates"": [ -119.66065...",Cathy,4,-119.660656,40.638609,-72.646741,42.230068,5227.743247,5227.212960
4,"Cleveland-Elyria, OH Metro","{ ""type"": ""Point"", ""coordinates"": [ -81.683882...",Cathy,5,-81.683882,41.375258,-119.660656,40.638609,4222.851845,4222.763351
5,"Knoxville, TN Metro","{ ""type"": ""Point"", ""coordinates"": [ -84.138485...",Cathy,6,-84.138485,36.047549,-81.683882,41.375258,282.561095,282.272338
6,"Madison, WI Metro","{ ""type"": ""Point"", ""coordinates"": [ -89.591695...",Cathy,7,-89.591695,43.078509,-84.138485,36.047549,607.874730,606.736061
7,"Phoenix-Mesa-Scottsdale, AZ Metro","{ ""type"": ""Point"", ""coordinates"": [ -112.07055...",Dan,1,-112.070554,33.185508,-89.591695,43.078509,2508.068407,2498.871539
8,"Miami-Fort Lauderdale-West Palm Beach, FL Metro","{ ""type"": ""Point"", ""coordinates"": [ -80.497102...",Dan,2,-80.497102,26.129562,-112.070554,33.185508,3511.857801,3505.086886
9,"Lexington-Fayette, KY Metro","{ ""type"": ""Point"", ""coordinates"": [ -84.432749...",Dan,3,-84.432749,38.090933,-80.497102,26.129562,471.099307,468.796095


In [ ]:
# filter out TripOrder = 1, group by REP, sum distance columns
df_dist = df[df["TripOrder"] != 1][["REP", "distance1", "distance2"]].groupby("REP").sum().sort_values("distance2", ascending=False)
# convert to miles
df_dist["distance1"] = df_dist["distance1"] / 1.609
df_dist["distance2"] = df_dist["distance2"] / 1.609
df_dist

,distance1,distance2
REP,,
Diana,11435.253066,11405.035188
Cathy,8306.775129,8305.148240
James,7455.814497,7450.296338
Dan,6318.404027,6309.597000
Steve,4850.100635,4844.448910
Scott,4626.959967,4625.699696
Mark,3092.291318,3090.504845


In [166]:
df_output

,REP,LengthMi
0,Diana,9077.844095
1,Cathy,6777.003786
2,Dan,6220.995173
3,James,6157.018626
4,Steve,4250.389398
5,Scott,4178.885363
6,Mark,3210.917267
